In [1]:
#importacao inicial
import pandas as pd
import numpy as np

df_dados = pd.read_excel('../data/raw/Dados Bibliograficos UEM_UZ_ISPQ 2025_2026.xls')
df_dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 34178 entries, 0 to 34177
Data columns (total 45 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Direct               0 non-null      float64       
 1   Ordem                0 non-null      float64       
 2   Ano                  34178 non-null  int64         
 3   NoLido               0 non-null      float64       
 4   candidato_codigo     34178 non-null  int64         
 5   apelido              34177 non-null  str           
 6   nome                 34177 non-null  str           
 7   TipoDoc              34178 non-null  str           
 8   DocIdent             34178 non-null  str           
 9   Sexo                 34177 non-null  str           
 10  EstCivil             34177 non-null  str           
 11  pai_nascimento       34176 non-null  str           
 12  ProvNasc             34176 non-null  str           
 13  ProvRes              34176 non-null  str  

In [2]:
df_dados.rename(columns={'pai_nascimento': 'pais_nascimento'}, inplace=True)

In [3]:
print(df_dados[df_dados['ProvNasc'] == 'Luanda'][['ProvNasc', 'distrito_nascimento', 'pais_nascimento']])

df_dados.loc[df_dados['ProvNasc'] == 'Luanda', ['ProvNasc', 'distrito_nascimento']] = 'Estrangeiro'

df_dados['ProvNasc'].value_counts()

      ProvNasc distrito_nascimento pais_nascimento
12776   Luanda         Estrangeiro          Angola


ProvNasc
Cidade de Maputo       12567
Província de Maputo     5645
Sofala                  4783
Inhambane               2457
Gaza                    2255
Zambezia                2098
Nampula                 1333
Manica                  1238
Tete                     863
Cabo Delgado             437
Niassa                   373
Estrangeiro              127
Name: count, dtype: int64

In [4]:
def corrigir_encoding(texto):
    """Reverte mojibake: recodifica como cp1252 e decodifica como utf-8"""
    if not isinstance(texto, str):
        return texto
    try:
        return texto.encode('cp1252').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return texto 

colunas_texto = df_dados.select_dtypes(include='object').columns
for col in colunas_texto:
    df_dados[col] = df_dados[col].apply(corrigir_encoding)

C:\Users\belci\AppData\Local\Temp\ipykernel_22044\1720817448.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df_dados.select_dtypes(include='object').columns


In [5]:

for col in colunas_texto:
    suspeitos = df_dados[col].astype(str).str.contains('Ã|‡', na=False, regex=True)
    if suspeitos.any():
        print(f"{col}: {suspeitos.sum()} valores ainda suspeitos")
        print(df_dados.loc[suspeitos, col].unique()[:5])

apelido: 427 valores ainda suspeitos
<ArrowStringArray>
['SEBASTIÃO', 'SABÃO', 'FERRÃO', 'SALOMÃO', 'ROMÃO']
Length: 5, dtype: str
nome: 1561 valores ainda suspeitos
<ArrowStringArray>
[              'FLORENTINA JOÃO',                   'MANUEL JOÃO',
 'SHELTON DA CELIA ESTEVÃO LUIS',                'CELESTINO JOÃO',
                   'AMÉLIA JOÃO']
Length: 5, dtype: str
TipoDoc: 876 valores ainda suspeitos
<ArrowStringArray>
[                             'CARTÃO DE ELEITOR',
                              'CARTA DE CONDUÇÃO',
                                    'TALÃO DE BI',
 'CARTÃO DE IDENTIFICAÇÃO DE REQUERENTE DE ASILO']
Length: 4, dtype: str


In [6]:
#colunas  irrelevantes
colunasRemover = [
    "Direct", "Ordem", "NoLido", "Dispensa", "RevProva",
    "celular", "celularAlternativo", "DataReg", "HoraReg", "OperReg"
]
df_dados.drop(columns=colunasRemover, inplace=True)
df_dados.head()

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,ISPQ_Opc1,ISPQ_Cod_Opc2,ISPQ_Opc2,UZ_Cod_Opc1,UZ_Opc1,UZ_Cod_Opc2,UZ_Opc2,Status,nuit,data_Nasc
0,2026,10019,SUARES,SONIA,BILHETE DE IDENTIDADE,1233444,FEMININO,Solteiro(a),Andorra,Estrangeiro,...,Engenharia de Aquacultura,80104.0,Engenharia de Processamento e Controlo de Qual...,35100.0,Administração Pública (Beira) – Diurno - UZ,35110.0,Economia (Beira) - Diurno - UZ,NaN,344553521.0,2011-12-11
1,2026,10039,MUARAPAZ,CLÁUDIO ARMANDO,BILHETE DE IDENTIDADE,030106033801F,MASCULINO,Solteiro(a),Mocambique,Nampula,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,177648922.0,2004-05-07
2,2026,10041,MBALANGO,YUNI ADELAIDE ANÍBAL,BILHETE DE IDENTIDADE,110107852412P,FEMININO,Solteiro(a),Mocambique,Cidade de Maputo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131513240.0,2007-10-13
3,2026,10047,CHALE,IBRAIMO OMAR,BILHETE DE IDENTIDADE,070102840008F,MASCULINO,Solteiro(a),Mocambique,Sofala,...,NaN,NaN,NaN,35213.0,Engenharia Mecatrónica (Beira) - Nocturno - UZ,35209.0,Engenharia Eléctrica (Beira) - Nocturno - UZ,NaN,123024631.0,1995-07-05
4,2026,10050,CUVACA,ANTONIO DOMINGOS,BILHETE DE IDENTIDADE,070108883250D,MASCULINO,Solteiro(a),Mocambique,Sofala,...,NaN,NaN,NaN,35210.0,Engenharia Informática (Beira) - Diurno - UZ,35208.0,Engenharia Eléctrica (Beira) - Diurno - UZ,NaN,174122997.0,2004-07-13


In [7]:
colunas_uem = ['UEM_Cod_Opc1', 'UEM_Opc1', 'UEM_Cod_Opc2', 'UEM_Opc2']
df_dados.dropna(subset=colunas_uem, thresh=1, inplace=True)
df_dados.shape

(26798, 35)

In [8]:
lista2 = ["ISPQ_Opc1", "ISPQ_Cod_Opc2", "UZ_Cod_Opc1", "UZ_Opc1", "ISPQ_Opc2", "UZ_Opc2", "Status", "ISPQ_Cod_Opc1", "UZ_Cod_Opc2", "nuit"]
df_dados.drop(columns=lista2, inplace=True)

  


In [9]:
#Copia da base de dados original 
df_analise = df_dados.copy()
#padronizacao de colunas
df_analise['UEM_Cod_Opc2'] = df_analise['UEM_Cod_Opc2'].fillna(0).astype(int)
df_analise['UEM_Opc2'] = df_analise['UEM_Opc2'].fillna('Sem Opcao')
df_analise['UEM_Cod_Opc1'] = df_analise['UEM_Cod_Opc1'].astype(int)
#padronizacao de colunas de texto
colunas_texto = [
    'UEM_Opc1', 'UEM_Opc2', 'EscoPU', 'local_exame',
    'distrito_nascimento', 'distrito_residencia',
    'Sexo', 'EstCivil', 'ProvNasc', 'pais_nascimento'
]
for col in colunas_texto:
    df_analise[col] = df_analise[col].str.strip().str.title()

df_analise.sample(10)

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,TipoEst,AnocPU,Classif,distrito_nascimento,distrito_residencia,UEM_Cod_Opc1,UEM_Opc1,UEM_Cod_Opc2,UEM_Opc2,data_Nasc
31773,2026,61968,MATEUS,MAMITA LUCAS,BILHETE DE IDENTIDADE,041408876229A,Feminino,Solteiro(A),Mocambique,Zambezia,...,NaN,2025,NaN,Namarroi,Kamaxaquene,11116,Engenharia De Petróleo E Gás Natural - Diurno ...,11104,Engenharia Eléctrica - Diurno - Uem,2005-09-28
29925,2026,35360,MANHICA,SILENCIA JÚLIO,TALÃO DE BI,233187001105125,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2018,NaN,Kamaxaquene,Kamaxaquene,10300,Economia - Diurno - Uem,10302,Contabilidade E Finanças - Diurno - Uem,2000-07-05
3786,2026,18008,VENDO,AMADE MATE ORLANDO,BILHETE DE IDENTIDADE,070108934142D,Masculino,Solteiro(A),Mocambique,Sofala,...,NaN,2024,NaN,Cidade Da Beira,Cidade Da Beira,11102,Engenharia Electrónica - Diurno - Uem,11106,Engenharia Mecânica - Diurno - Uem,2007-06-12
32278,2026,62952,MACUCULE,VALENTIM LEONARDO,BILHETE DE IDENTIDADE,080106500363N,Masculino,Solteiro(A),Mocambique,Manica,...,NaN,2025,NaN,Chimoio Cidade,Inhambane (Cidade),11104,Engenharia Eléctrica - Diurno - Uem,11100,Engenharia Civil - Diurno - Uem,2006-11-03
21940,2026,50462,ALFAICA,FÁBIO BRUNO FIEL,BILHETE DE IDENTIDADE,040107763553C,Masculino,Solteiro(A),Mocambique,Zambezia,...,NaN,2025,NaN,Cidade De Quelimane,Cidade De Quelimane,11104,Engenharia Eléctrica - Diurno - Uem,11116,Engenharia De Petróleo E Gás Natural - Diurno ...,2008-07-26
15891,2026,41976,MACHAVANE,ATÁLIA FRANCISCO,BILHETE DE IDENTIDADE,110107076534S,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2024,NaN,Kamavota,Kamavota,10100,Administração Pública - Diurno - Uem,10122,Sociologia - Diurno - Uem,2006-03-03
31637,2026,61711,MACIE,NÉRCIA NÉRCIO,BILHETE DE IDENTIDADE,100107778095M,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2024,NaN,Kamavota,Kamavota,10500,Jornalismo - Diurno - Uem,10506,Biblioteconomia (Por-I E His-I) - Diurno - Uem,2006-09-23
13567,2026,28393,USSIVANE,DÉRCIO XADREQUE,BILHETE DE IDENTIDADE,110107602364S,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2023,NaN,Kanlhamankulu,Kamubukwana,11000,Informática (Mat-Ii E Por-Ii) - Diurno - Uem,11004,Ciências De Infor. Geog (Mat-Ii E Por-Ii) - Di...,2006-05-08
2376,2026,15927,MANGAZE,GEOVANNI JORGE PAIS,BILHETE DE IDENTIDADE,100100652778Q,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,NaN,2025,NaN,Kampfumo,Matola Cidade,10600,Direito - Diurno - Uem,10300,Economia - Diurno - Uem,2008-08-14
13570,2026,38056,SITOE,FREDSON JOAO,BILHETE DE IDENTIDADE,010104685938B,Masculino,Solteiro(A),Mocambique,Niassa,...,NaN,2024,NaN,Lichinga,Katembe,11035,Hidrogeologia E Recursos Hídricos - Nocturno -...,11029,Geo-Ciências De Petróleo E Gás - Nocturno - Uem,2006-11-16


In [10]:
lista3 = ['TipoEst', 'Classif']
df_analise.drop(columns=lista3, inplace=True)
display(df_analise.sample(10))
display(df_analise.info())

,Ano,candidato_codigo,apelido,nome,TipoDoc,DocIdent,Sexo,EstCivil,pais_nascimento,ProvNasc,...,EscoPU,local_exame,AnocPU,distrito_nascimento,distrito_residencia,UEM_Cod_Opc1,UEM_Opc1,UEM_Cod_Opc2,UEM_Opc2,data_Nasc
17917,2026,45066,BASTOS,SERGIO ABERTO,BILHETE DE IDENTIDADE,110208877053F,Masculino,Solteiro(A),Mocambique,Província De Maputo,...,Escola Secundária Heróis Moçambicanos,Cidade De Maputo,2025,Matola Cidade,Matola Cidade,10600,Direito - Diurno - Uem,10102,Antropologia - Diurno - Uem,2005-03-13
29659,2026,59457,CHAINCOMO,DORCA ABDALA,BILHETE DE IDENTIDADE,110108018118C,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Escola Secundária Inhaca Sede,Cidade De Maputo,2023,Kanyaka,Kanyaka,10208,Desenvolvimento E Educação De Infância - Diurn...,10204,Educação Ambiental (Por-Iii E Bio-Ii) - Diurno...,2003-09-13
28371,2026,57967,MATOLA,TANIA AIDA,BILHETE DE IDENTIDADE,110104402303P,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Escola Secundária Da Matola,Cidade De Maputo,2025,Kampfumo,Boane,10305,Gestão - Nocturno - Uem,11003,Estatística (Mat-Ii E Por-Ii) - Nocturno - Uem,2007-05-22
11527,2026,32191,MOMADE,JAMAL ISMAEL,BILHETE DE IDENTIDADE,110106445076Q,Masculino,Solteiro(A),Mocambique,Cidade De Maputo,...,Escola Secundária Noroeste 1,Cidade De Maputo,2025,Kanlhamankulu,Kamaxaquene,10800,Medicina - Diurno - Uem,11008,Biologia E Saúde - Diurno - Uem,2008-11-11
33371,2026,64368,MACHOCO,SANDRA AMADEU,BILHETE DE IDENTIDADE,110102710016N,Feminino,Solteiro(A),Mocambique,Cidade De Maputo,...,Escola Secundária Armando Emílio Guebuza,Cidade De Maputo,2024,Kampfumo,Matola Cidade,10206,Psicologia Escolar E De Necessidades Educativa...,10208,Desenvolvimento E Educação De Infância - Diurn...,1992-07-02
9067,2026,25830,MUTEMBA,FILORDA GILBERTO,BILHETE DE IDENTIDADE,100107198762D,Feminino,Solteiro(A),Mocambique,Província De Maputo,...,Escola Secundária Da Zona-Verde,Cidade De Maputo,2024,Matola Cidade,Matola Cidade,10600,Direito - Diurno - Uem,10100,Administração Pública - Diurno - Uem,2007-07-18
32567,2026,63363,MACULUVE,BRAITON ZEFANIA ISAC,BILHETE DE IDENTIDADE,110104836565N,Masculino,Casado(A),Mocambique,Tete,...,Escola Secundária Gerardo Gumiero,Cidade De Maputo,2006,Changara,Matola Cidade,10127,Tradução Português/Inglês - Nocturno - Uem,11003,Estatística (Mat-Ii E Por-Ii) - Nocturno - Uem,1988-01-29
28635,2026,58361,HERMINIO,HERCIA FILOMENA DE ISSA,BILHETE DE IDENTIDADE,031706837506s,Feminino,Solteiro(A),Mocambique,Nampula,...,Estrangeira,Chongoene,2025,Nacala-Porto,Cidade De Xai-Xai,11100,Engenharia Civil - Diurno - Uem,11116,Engenharia De Petróleo E Gás Natural - Diurno ...,2005-09-08
7473,2026,26430,DUARTE,IVAN GONCALVES,PASSAPORTE,110108910990F,Masculino,Solteiro(A),Mocambique,Província De Maputo,...,Outra - Província De Maputo,Cidade De Maputo,2025,Matola Cidade,Matola Cidade,11200,Arquitectura E Planeamento Físico - Diurno - Uem,11110,Engenharia Informática - Diurno - Uem,2007-12-04
32860,2026,63729,MASSINGUE,CARLOS,BILHETE DE IDENTIDADE,100108913991J,Masculino,Solteiro(A),Mocambique,Província De Maputo,...,Outra - Província De Maputo,Cidade De Maputo,2025,Matola Cidade,Boane,11106,Engenharia Mecânica - Diurno - Uem,11102,Engenharia Electrónica - Diurno - Uem,2006-12-12


<class 'pandas.DataFrame'>
Index: 26798 entries, 0 to 34177
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Ano                  26798 non-null  int64         
 1   candidato_codigo     26798 non-null  int64         
 2   apelido              26797 non-null  str           
 3   nome                 26797 non-null  str           
 4   TipoDoc              26798 non-null  str           
 5   DocIdent             26798 non-null  str           
 6   Sexo                 26797 non-null  str           
 7   EstCivil             26797 non-null  str           
 8   pais_nascimento      26797 non-null  str           
 9   ProvNasc             26797 non-null  str           
 10  ProvRes              26797 non-null  str           
 11  ProvCand             26798 non-null  str           
 12  cod_preUni           26769 non-null  float64       
 13  EscoPU               26798 non-null  str       

None

In [11]:
df_analise.to_parquet('../data/processed/candidatos_uem.parquet', index=False)
print("Guardado:", df_analise.shape)

Guardado: (26798, 23)
